# GPT Language Model - 修复版 V2

## ⚠️ 重要：运行前必须执行以下步骤
1. **重启Kernel**: Kernel → Restart Kernel (清除CUDA错误状态)
2. **删除旧模型**: 删除 `model-01.pkl` 文件（如果存在）
3. **按顺序运行所有Cell**

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import mmap
import random
import pickle
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ============ 超参数 ============
block_size = 128
batch_size = 64
max_iters = 200
learning_rate = 3e-4
eval_iters = 100
n_embd = 384
n_head = 4
n_layer = 4
dropout = 0.3

print(f"使用设备: {device}")
print(f"block_size: {block_size}")

In [ ]:
# ============ 加载词汇表 ============
chars = ""
with open("vocab.txt", 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(set(text))
vocab_size = len(chars)
print(f"词汇表大小: {vocab_size}")
print(f"前10个字符: {chars[:10]}")

In [ ]:
# ============ 编码/解码器 (修复版) ============
string_to_int = { ch:i for i, ch in enumerate(chars) }
int_to_string = { i:ch for i, ch in enumerate(chars) }

# ✅ 修复：安全的编码函数，跳过未知字符
def encode(s):
    """将字符串编码为整数列表，跳过不在词汇表中的字符"""
    result = []
    for c in s:
        if c in string_to_int:
            result.append(string_to_int[c])
        # 跳过未知字符，避免KeyError
    return result

def decode(l):
    """将整数列表解码为字符串"""
    return ''.join([int_to_string.get(i, '?') for i in l])

# 测试
print("编码测试: 'hello' ->", encode('hello'))
print("解码测试:", decode(encode('hello')))

In [ ]:
# ============ 数据加载 (修复版) ============

def get_random_chunk(split):
    data_dir = os.environ.get("GPT_DATA_DIR", "data_samples")
    filename = os.path.join(data_dir, "train_split.txt" if split == 'train' else "val_split.txt")
    with open(filename, "rb") as f:
        with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mm:
            file_size = len(mm)
            # 确保读取足够的数据
            chunk_size = block_size * batch_size * 2  # 多读一些以确保足够
            start_pos = random.randint(0, max(0, file_size - chunk_size))
            mm.seek(start_pos)
            block = mm.read(chunk_size)
            decoded_block = block.decode('utf-8', errors='ignore').replace('\r', ' ')
            
            # 使用安全的编码函数
            encoded = encode(decoded_block)
            data = torch.tensor(encoded, dtype=torch.long)
    return data

def get_batch(split):
    data = get_random_chunk(split)
    
    # ✅ 安全检查：确保数据长度足够
    min_length = block_size + 1
    retry_count = 0
    while len(data) < min_length and retry_count < 5:
        data = get_random_chunk(split)
        retry_count += 1
    
    if len(data) < min_length:
        raise ValueError(f"数据长度不足: {len(data)} < {min_length}")
    
    # 确保不会越界
    max_start = len(data) - block_size - 1
    if max_start <= 0:
        max_start = 1
    
    ix = torch.randint(0, max_start, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

# 测试数据加载
print("测试数据加载...")
try:
    test_x, test_y = get_batch('train')
    print(f"✅ 数据加载成功!")
    print(f"   x shape: {test_x.shape}")
    print(f"   y shape: {test_y.shape}")
    print(f"   x max value: {test_x.max().item()} (应 < {vocab_size})")
    print(f"   y max value: {test_y.max().item()} (应 < {vocab_size})")
except Exception as e:
    print(f"❌ 数据加载失败: {e}")

In [ ]:
# ============ 损失评估函数 ============
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
# ============ 模型定义 (修复版) ============

class Head(nn.Module):
    """单个注意力头"""
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out


class MultiHeadAttention(nn.Module):
    """多头注意力"""
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out


class FeedForward(nn.Module):
    """前馈网络"""
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Transformer Block"""
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        y = self.sa(x)
        x = self.ln1(x + y)
        y = self.ffwd(x)
        x = self.ln2(x + y)
        return x


class GPTLanguageModel(nn.Module):
    """GPT语言模型"""
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        B, T = index.shape
        
        # ✅ 安全检查：确保T不超过block_size
        if T > block_size:
            index = index[:, -block_size:]  # 只取最后block_size个token
            T = block_size
            if targets is not None:
                targets = targets[:, -block_size:]
        
        tok_emb = self.token_embedding_table(index)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)  # ✅ 修复: 原来是 x - self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            # ✅ 截断输入以适应block_size，防止位置嵌入越界
            index_cond = index[:, -block_size:]
            logits, loss = self.forward(index_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=-1)
        return index


print("✅ 模型类定义完成！")

In [ ]:
# ============ 创建模型 (从头开始训练) ============

# ✅ 重要：不加载旧模型，从头开始训练
model = GPTLanguageModel(vocab_size)
m = model.to(device)

# 打印模型信息
total_params = sum(p.numel() for p in model.parameters())
print(f"✅ 新模型创建成功！")
print(f"   模型参数量: {total_params:,} ({total_params/1e6:.2f}M)")
print(f"   vocab_size: {vocab_size}")
print(f"   block_size: {block_size}")

In [ ]:
# ============ 快速测试模型 ============
print("测试模型前向传播...")
try:
    test_x, test_y = get_batch('train')
    logits, loss = model(test_x, test_y)
    print(f"✅ 模型测试成功!")
    print(f"   logits shape: {logits.shape}")
    print(f"   loss: {loss.item():.4f}")
except Exception as e:
    print(f"❌ 模型测试失败: {e}")
    print("请重启Kernel后重新运行所有Cell")

In [ ]:
# ============ 训练循环 ============

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# 记录最佳模型
best_val_loss = float('inf')

print("开始训练...")
print("="*50)

for iter in range(max_iters):
    # 评估损失
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step {iter:4d}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        
        # 保存最佳模型
        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            with open('model-best.pkl', 'wb') as f:
                pickle.dump(model, f)
            print(f"         → 保存最佳模型 (val_loss: {best_val_loss:.4f})")
    
    # 获取批次
    xb, yb = get_batch('train')
    
    # 前向传播
    logits, loss = model(xb, yb)
    
    # 反向传播
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("="*50)
print(f"训练完成！最终loss: {loss.item():.4f}")

# 保存最终模型
with open('model-01.pkl', 'wb') as f:
    pickle.dump(model, f)
print('模型已保存到 model-01.pkl')

In [ ]:
# ============ 文本生成 ============

print("生成文本...")
print("="*50)

# 从一个空token开始生成
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)
print("="*50)

In [ ]:
# ============ 带提示词的文本生成 ============

prompt = "The meaning of life is"
print(f"提示词: '{prompt}'")
print("="*50)

# 编码提示词
prompt_encoded = encode(prompt)
context = torch.tensor([prompt_encoded], dtype=torch.long, device=device)

# 生成
generated = m.generate(context, max_new_tokens=200)[0].tolist()
print(decode(generated))
print("="*50)